# Lab 2: Word Embeddings & Named Entity Recognition

## Overview
This notebook implements two fundamental NLP tasks from scratch using PyTorch:

1. **Word2Vec Skip-Gram with Negative Sampling** — learning word representations from the CoNLL-2003 corpus
2. **Named Entity Recognition (NER)** — comparing a Feed-Forward Neural Network against a Hidden Markov Model with Viterbi decoding

**Dataset:** CoNLL-2003 (PER, ORG, LOC, MISC entities)

In [ ]:
# Install dependencies (uncomment if needed)
# !pip install datasets torch scikit-learn matplotlib numpy -q

In [ ]:
import datasets
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from collections import Counter
from sklearn.manifold import TSNE
from sklearn.metrics import precision_recall_fscore_support, classification_report
from sklearn.metrics.pairwise import cosine_similarity
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import random

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
print("Imports OK")

---
## Part 1: Word2Vec Skip-Gram with Negative Sampling

### 1.1 Load Dataset

In [ ]:
dataset = datasets.load_dataset("lhoestq/conll2003")

ner_classes = dataset['train'].features['ner_tags'].feature.names
num_ner_classes = len(ner_classes)
print(f"NER classes ({num_ner_classes}): {ner_classes}")
print(f"Train / Val / Test: {len(dataset['train'])} / {len(dataset['validation'])} / {len(dataset['test'])} sentences")

### 1.2 Preprocessing

In [ ]:
def preprocess_data(dataset):
    """Extract and lowercase all sentences from all splits for Word2Vec training."""
    sentences = []
    for split in ['train', 'validation', 'test']:
        for tokens in dataset[split]['tokens']:
            lowered = [t.lower() for t in tokens if t.strip()]
            if len(lowered) > 1:
                sentences.append(lowered)
    return sentences

sentences = preprocess_data(dataset)
print(f"Total sentences: {len(sentences)}")
print(f"Sample: {sentences[0]}")

### 1.3 Vocabulary

In [ ]:
class Vocabulary:
    def __init__(self, sentences, min_freq=2):
        self.word_freq = Counter(w for s in sentences for w in s)
        valid = [w for w, f in self.word_freq.items() if f >= min_freq]
        self.word2idx = {'<PAD>': 0, '<UNK>': 1}
        self.idx2word = {0: '<PAD>', 1: '<UNK>'}
        for idx, word in enumerate(valid, start=2):
            self.word2idx[word] = idx
            self.idx2word[idx] = word
        print(f"Vocabulary size: {len(self.word2idx)}")

    def __len__(self):
        return len(self.word2idx)

    def __getitem__(self, word):
        return self.word2idx.get(word, 1)

vocab = Vocabulary(sentences, min_freq=2)

### 1.4 Skip-Gram Training Pairs

In [ ]:
def generate_training_data(sentences, vocab, window_size=2):
    """Generate (center, context) index pairs for Skip-Gram training."""
    pairs = []
    for sentence in sentences:
        indices = [vocab[w] for w in sentence if vocab[w] != 1]
        for i, center in enumerate(indices):
            for j in range(max(0, i - window_size), min(len(indices), i + window_size + 1)):
                if j != i:
                    pairs.append((center, indices[j]))
    return pairs

training_pairs = generate_training_data(sentences, vocab, window_size=2)
print(f"Training pairs: {len(training_pairs):,}")

### 1.5 Negative Sampler

In [ ]:
class NegativeSampler:
    """Samples negative words proportional to frequency^(3/4) as in the original Word2Vec paper."""
    def __init__(self, vocab, num_negative=5):
        self.num_negative = num_negative
        freq = np.array([vocab.word_freq.get(vocab.idx2word[i], 0) for i in range(len(vocab))])
        freq[0] = freq[1] = 0  # exclude PAD and UNK
        probs = freq ** 0.75
        self.probs = probs / probs.sum()
        self.indices = np.arange(len(vocab))

    def sample(self, batch_size):
        return np.random.choice(self.indices, size=(batch_size, self.num_negative), p=self.probs)

### 1.6 Skip-Gram Model

In [ ]:
class SkipGramNeg(nn.Module):
    """Skip-Gram model trained with Negative Sampling (Mikolov et al., 2013)."""
    def __init__(self, vocab_size, embedding_dim=100):
        super().__init__()
        self.center  = nn.Embedding(vocab_size, embedding_dim)
        self.context = nn.Embedding(vocab_size, embedding_dim)
        nn.init.uniform_(self.center.weight,  -0.5 / embedding_dim, 0.5 / embedding_dim)
        nn.init.uniform_(self.context.weight, -0.5 / embedding_dim, 0.5 / embedding_dim)

    def forward(self, center_words, context_words, negative_words):
        # Positive loss
        c_emb = self.center(center_words)                        # (B, D)
        ctx_emb = self.context(context_words)                    # (B, D)
        pos_loss = -F.logsigmoid((c_emb * ctx_emb).sum(1)).mean()

        # Negative loss
        neg_emb = self.context(negative_words)                   # (B, K, D)
        neg_scores = (c_emb.unsqueeze(1) * neg_emb).sum(2)      # (B, K)
        neg_loss = -F.logsigmoid(-neg_scores).mean()

        return pos_loss + neg_loss

    def get_embeddings(self):
        """Return averaged center + context embeddings as final word vectors."""
        return (self.center.weight.data + self.context.weight.data) / 2

### 1.7 Training

In [ ]:
def train_word2vec(pairs, vocab, emb_dim=100, batch_size=1024,
                   num_negative=5, lr=0.025, epochs=15):
    model   = SkipGramNeg(len(vocab), emb_dim)
    opt     = optim.Adam(model.parameters(), lr=lr)
    sampler = NegativeSampler(vocab, num_negative)
    losses  = []

    for epoch in range(epochs):
        random.shuffle(pairs)
        total, n_batches = 0.0, 0
        for i in range(0, len(pairs), batch_size):
            batch = pairs[i : i + batch_size]
            if len(batch) < 2:
                continue
            centers   = torch.LongTensor([p[0] for p in batch])
            contexts  = torch.LongTensor([p[1] for p in batch])
            negatives = torch.LongTensor(sampler.sample(len(batch)))

            loss = model(centers, contexts, negatives)
            opt.zero_grad(); loss.backward(); opt.step()
            total += loss.item(); n_batches += 1

        avg = total / n_batches
        losses.append(avg)
        print(f"Epoch {epoch+1:>2}/{epochs}  loss={avg:.4f}")

    return model, losses

print("Training Word2Vec …")
w2v_model, losses = train_word2vec(training_pairs, vocab)

In [ ]:
plt.figure(figsize=(9, 4))
plt.plot(range(1, len(losses)+1), losses, marker='o', linewidth=2)
plt.title('Word2Vec Training Loss'); plt.xlabel('Epoch'); plt.ylabel('Loss')
plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig('word2vec_training_loss.png', dpi=150)
plt.show()

### 1.8 Save & Load Embeddings

In [ ]:
def save_embeddings(model, vocab, path='word_embeddings.npy'):
    emb = model.get_embeddings().cpu().numpy()
    np.save(path, {'embeddings': emb, 'word2idx': vocab.word2idx, 'idx2word': vocab.idx2word},
            allow_pickle=True)
    print(f"Saved {emb.shape} embeddings → {path}")

def load_embeddings(path='word_embeddings.npy'):
    d = np.load(path, allow_pickle=True).item()
    return d['embeddings'], d['word2idx'], d['idx2word']

save_embeddings(w2v_model, vocab)
embeddings, word2idx, idx2word = load_embeddings()
print(f"Embedding matrix: {embeddings.shape}")

### 1.9 Word Similarity & Analogies

In [ ]:
def find_similar_words(embeddings, word2idx, idx2word, target, top_k=10):
    if target not in word2idx:
        print(f"'{target}' not in vocabulary"); return []
    tidx = word2idx[target]
    tvec = embeddings[tidx].reshape(1, -1)
    sims = [(idx2word[i], cosine_similarity(tvec, embeddings[i].reshape(1,-1))[0][0])
            for i in range(len(embeddings)) if idx2word[i] not in ('<PAD>','<UNK>') and i != tidx]
    sims.sort(key=lambda x: x[1], reverse=True)
    print(f"Similar to '{target}':", [f"{w}({s:.2f})" for w, s in sims[:top_k]])
    return sims[:top_k]

def word_analogy(embeddings, word2idx, idx2word, a, b, c, top_k=5):
    for w in (a, b, c):
        if w not in word2idx: print(f"'{w}' not in vocabulary"); return
    target = embeddings[word2idx[b]] - embeddings[word2idx[a]] + embeddings[word2idx[c]]
    excl = {word2idx[a], word2idx[b], word2idx[c]}
    sims = [(idx2word[i], cosine_similarity(target.reshape(1,-1), embeddings[i].reshape(1,-1))[0][0])
            for i in range(len(embeddings)) if i not in excl and idx2word[i] not in ('<PAD>','<UNK>')]
    sims.sort(key=lambda x: x[1], reverse=True)
    print(f"{a}:{b} :: {c}:?  → {[w for w,_ in sims[:top_k]]}")
    return sims[:top_k]

print("=" * 55)
find_similar_words(embeddings, word2idx, idx2word, 'government')
find_similar_words(embeddings, word2idx, idx2word, 'london')
print()
word_analogy(embeddings, word2idx, idx2word, 'london', 'england', 'paris')
word_analogy(embeddings, word2idx, idx2word, 'walk', 'walking', 'swim')

### 1.10 t-SNE Visualization

In [ ]:
def visualize_embeddings(embeddings, word2idx, words, title='Word Embeddings (t-SNE)'):
    valid = [w for w in words if w in word2idx]
    if len(valid) < 2: print("Not enough words"); return
    vecs = embeddings[[word2idx[w] for w in valid]]
    coords = TSNE(n_components=2, random_state=42, perplexity=min(30, len(valid)-1)).fit_transform(vecs)

    plt.figure(figsize=(14, 10))
    plt.scatter(coords[:, 0], coords[:, 1], alpha=0.6, s=60)
    for i, w in enumerate(valid):
        plt.annotate(w, coords[i], xytext=(5,5), textcoords='offset points', fontsize=9)
    plt.title(title, fontsize=14); plt.grid(alpha=0.3); plt.tight_layout()
    plt.savefig('word_embeddings_tsne.png', dpi=150, bbox_inches='tight')
    plt.show()

words_to_plot = [
    'england','france','germany','italy','spain',
    'london','paris','berlin','rome','madrid',
    'president','minister','director','manager',
    'company','government','organization','committee',
    'the','and','of','to','in',
]
visualize_embeddings(embeddings, word2idx, words_to_plot)

---
## Part 2: Named Entity Recognition

We compare:
- **Feed-Forward NN** — token-level classifier using the trained Word2Vec embeddings
- **Hidden Markov Model** — statistical sequence model decoded with Viterbi

### 2.1 Extract NER Data

In [ ]:
def extract_ner_data(split):
    sents = [[t.lower() for t in s] for s in split['tokens']]
    tags  = [list(t) for t in split['ner_tags']]
    return sents, tags

train_sents, train_tags = extract_ner_data(dataset['train'])
val_sents,   val_tags   = extract_ner_data(dataset['validation'])
test_sents,  test_tags  = extract_ner_data(dataset['test'])

print(f"Train {len(train_sents)} / Val {len(val_sents)} / Test {len(test_sents)}")
print(f"Classes: {ner_classes}")

### 2.2 NERDataset

In [ ]:
class NERDataset(Dataset):
    def __init__(self, split, word2idx, embeddings, max_len=128):
        self.emb = embeddings
        self.data = []
        for tokens, tags in zip(split['tokens'], split['ner_tags']):
            if not (1 < len(tokens) <= max_len): continue
            idx  = [word2idx.get(t.lower(), 1) for t in tokens]   # 1 = UNK
            padded_idx  = (idx  + [0]*max_len)[:max_len]
            padded_tags = (list(tags) + [0]*max_len)[:max_len]
            self.data.append((padded_idx, padded_tags))

    def __len__(self): return len(self.data)

    def __getitem__(self, i):
        idx, tags = self.data[i]
        return torch.FloatTensor(self.emb[idx]), torch.LongTensor(tags)

train_ds = NERDataset(dataset['train'],      vocab.word2idx, embeddings)
val_ds   = NERDataset(dataset['validation'], vocab.word2idx, embeddings)
test_ds  = NERDataset(dataset['test'],       vocab.word2idx, embeddings)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=32)
test_loader  = DataLoader(test_ds,  batch_size=32)
print(f"Batches — train:{len(train_loader)} val:{len(val_loader)} test:{len(test_loader)}")

### 2.3 Feed-Forward NER Model

In [ ]:
class FeedForwardNER(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_classes, num_layers=2, dropout=0.3):
        super().__init__()
        layers = [nn.Linear(input_dim, hidden_dim), nn.ReLU(), nn.Dropout(dropout)]
        for _ in range(num_layers - 1):
            layers += [nn.Linear(hidden_dim, hidden_dim), nn.ReLU(), nn.Dropout(dropout)]
        self.net    = nn.Sequential(*layers)
        self.output = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):                   # x: (B, T, D)
        B, T, D = x.shape
        out = self.output(self.net(x.view(B*T, D)))
        return out.view(B, T, -1)

In [ ]:
def train_ffn(model, train_loader, val_loader, epochs=20, lr=1e-3):
    criterion = nn.CrossEntropyLoss(ignore_index=0)
    opt = optim.Adam(model.parameters(), lr=lr)
    train_hist, val_hist = [], []
    model = model.to(DEVICE)

    for epoch in range(epochs):
        model.train()
        t_loss = 0.0
        for emb, lbl in train_loader:
            emb, lbl = emb.to(DEVICE), lbl.to(DEVICE)
            opt.zero_grad()
            loss = criterion(model(emb).permute(0, 2, 1), lbl)
            loss.backward()
            opt.step()
            t_loss += loss.item()
        t_loss /= len(train_loader)

        model.eval()
        v_loss = 0.0
        with torch.no_grad():
            for emb, lbl in val_loader:
                emb, lbl = emb.to(DEVICE), lbl.to(DEVICE)
                v_loss += criterion(model(emb).permute(0, 2, 1), lbl).item()
        v_loss /= len(val_loader)

        train_hist.append(t_loss)
        val_hist.append(v_loss)
        print(f'Epoch {epoch+1:>2}/{epochs}  train={t_loss:.4f}  val={v_loss:.4f}')

    return train_hist, val_hist

emb_dim   = embeddings.shape[1]   # 100
ffn_model = FeedForwardNER(emb_dim, 256, num_ner_classes)
print(f'FFN parameters: {sum(p.numel() for p in ffn_model.parameters()):,}  device: {DEVICE}')
train_hist, val_hist = train_ffn(ffn_model, train_loader, val_loader)

In [ ]:
plt.figure(figsize=(9,4))
plt.plot(train_hist, label='Train'); plt.plot(val_hist, label='Val')
plt.title('FFN NER Loss'); plt.xlabel('Epoch'); plt.ylabel('Loss')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig('ffn_ner_loss.png', dpi=150); plt.show()

In [ ]:
def evaluate_ner(model, loader, ner_classes):
    model.eval()
    preds_all, labels_all = [], []
    with torch.no_grad():
        for emb, lbl in loader:
            emb, lbl = emb.to(DEVICE), lbl.to(DEVICE)
            preds = torch.argmax(model(emb), dim=-1)
            mask  = lbl != 0
            preds_all.extend(preds[mask].numpy())
            labels_all.extend(lbl[mask].numpy())

    present = sorted(set(labels_all + preds_all))
    p, r, f, _ = precision_recall_fscore_support(labels_all, preds_all, average='weighted', zero_division=0)
    print(f"Precision {p:.4f} | Recall {r:.4f} | F1 {f:.4f}")
    print(classification_report(labels_all, preds_all,
          labels=present, target_names=[ner_classes[i] for i in present], zero_division=0))
    return p, r, f

print("FFN evaluation on test set:")
ffn_p, ffn_r, ffn_f1 = evaluate_ner(ffn_model, test_loader, ner_classes)

### 2.4 Hidden Markov Model with Viterbi Decoding

In [ ]:
class HMM_NER:
    """First-order HMM for NER with Laplace smoothing and Viterbi decoding."""
    def __init__(self, smoothing=1e-5):
        self.alpha = smoothing

    def train(self, sentences, tags):
        tag_set  = sorted({t for seq in tags for t in seq})
        word_set = sorted({w for s in sentences for w in s})

        self.tag2i  = {t: i for i, t in enumerate(tag_set)}
        self.i2tag  = {i: t for t, i in self.tag2i.items()}
        self.word2i = {w: i for i, w in enumerate(word_set)}

        n_t, n_w = len(tag_set), len(word_set)
        trans = np.zeros((n_t, n_t))
        emit  = np.zeros((n_t, n_w))
        init  = np.zeros(n_t)
        tcnt  = np.zeros(n_t)
        n_s   = 0

        for sent, stags in zip(sentences, tags):
            if not sent: continue
            n_s += 1
            init[self.tag2i[stags[0]]] += 1
            for k, (w, t) in enumerate(zip(sent, stags)):
                ti = self.tag2i[t]
                tcnt[ti] += 1
                emit[ti, self.word2i[w]] += 1
                if k < len(stags)-1:
                    trans[ti, self.tag2i[stags[k+1]]] += 1

        self.init_p  = (init + self.alpha) / (n_s + self.alpha * n_t)

        self.trans_p = np.zeros((n_t, n_t))
        for i in range(n_t):
            row = trans[i] + self.alpha
            self.trans_p[i] = row / row.sum()

        self.emit_p = np.zeros((n_t, n_w))
        for i in range(n_t):
            row = emit[i] + self.alpha * 0.1
            self.emit_p[i] = row / row.sum()

        # Unknown-word probabilities: favour entity tags over O
        unk = np.zeros(n_t)
        for i in range(n_t):
            rare = emit[i] <= 1
            unk[i] = self.emit_p[i][rare].mean() if rare.sum() > 0 else 1e-6
        unk[1:] *= 2.0   # boost non-O tags
        self.unk_p = unk / unk.sum()
        print(f"HMM trained: {n_t} tags, {n_w} words, {n_s} sentences")

    def viterbi(self, sentence):
        if not sentence: return []
        n_t, T = len(self.tag2i), len(sentence)
        log_init  = np.log(self.init_p  + 1e-10)
        log_trans = np.log(self.trans_p + 1e-10)
        dp = np.full((n_t, T), -np.inf)
        bp = np.zeros((n_t, T), dtype=int)

        for s in range(n_t):
            wi = self.word2i.get(sentence[0], -1)
            ep = self.unk_p[s] if wi == -1 else self.emit_p[s, wi]
            dp[s, 0] = log_init[s] + np.log(ep + 1e-10)

        for t in range(1, T):
            wi = self.word2i.get(sentence[t], -1)
            for s in range(n_t):
                ep = self.unk_p[s] if wi == -1 else self.emit_p[s, wi]
                scores    = dp[:, t-1] + log_trans[:, s] + np.log(ep + 1e-10)
                bp[s, t]  = scores.argmax()
                dp[s, t]  = scores[bp[s, t]]

        path = [dp[:, T-1].argmax()]
        for t in range(T-1, 0, -1):
            path.append(bp[path[-1], t])
        path.reverse()
        return [self.i2tag[s] for s in path]

    def evaluate(self, sentences, tags):
        preds_all, true_all = [], []
        for sent, true in zip(sentences, tags):
            if not sent: continue
            pred = self.viterbi(sent)
            n    = min(len(pred), len(true))
            preds_all.extend(pred[:n])
            true_all.extend(str(x) for x in true[:n])
        p, r, f, _ = precision_recall_fscore_support(true_all, preds_all, average='weighted', zero_division=0)
        acc = np.mean([p == t for p, t in zip(preds_all, true_all)])
        print(f"Accuracy {acc:.4f} | Precision {p:.4f} | Recall {r:.4f} | F1 {f:.4f}")
        return p, r, f

hmm = HMM_NER()
hmm.train(train_sents, [list(map(str, t)) for t in train_tags])
print("\nHMM evaluation on test set:")
hmm_p, hmm_r, hmm_f1 = hmm.evaluate(test_sents, test_tags)

### 2.5 Model Comparison

In [ ]:
models     = ['Feed-Forward NN', 'HMM + Viterbi']
precisions = [ffn_p,  hmm_p]
recalls    = [ffn_r,  hmm_r]
f1_scores  = [ffn_f1, hmm_f1]

x, w = np.arange(2), 0.25
fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(x - w, precisions, w, label='Precision', alpha=0.85)
ax.bar(x,     recalls,    w, label='Recall',    alpha=0.85)
ax.bar(x + w, f1_scores,  w, label='F1',        alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(models)
ax.set_ylim(0, 1); ax.set_ylabel('Score')
ax.set_title('NER Model Comparison'); ax.legend(); ax.grid(alpha=0.3, axis='y')
plt.tight_layout(); plt.savefig('ner_comparison.png', dpi=150); plt.show()

print(f"\n{'Model':<22} {'Precision':>10} {'Recall':>10} {'F1':>8}")
print("-" * 55)
for m, p, r, f in zip(models, precisions, recalls, f1_scores):
    print(f"{m:<22} {p:>10.4f} {r:>10.4f} {f:>8.4f}")
best = models[0] if ffn_f1 >= hmm_f1 else models[1]
print(f"\nBetter model: {best}")

### 2.6 Sample Predictions

In [ ]:
sample = ["John", "Smith", "works", "for", "Google", "in", "London", "."]
lower  = [w.lower() for w in sample]

# FFN
ffn_model.eval()
with torch.no_grad():
    idx  = [vocab.word2idx.get(w, 1) for w in lower]
    emb  = torch.FloatTensor(embeddings[idx]).unsqueeze(0)
    ffn_preds = torch.argmax(ffn_model(emb), dim=-1)[0][:len(sample)].numpy()

# HMM
hmm_preds = hmm.viterbi(lower)

print(f"{'Token':<12} {'FFN':<14} {'HMM'}")
print("-" * 40)
for word, fp, hp in zip(sample, ffn_preds, hmm_preds):
    print(f"{word:<12} {ner_classes[fp]:<14} {hp}")